# Notebook 2 — Data Collection & Understanding
## Dataset: NSL-KDD Network Intrusion Detection

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_dataset, dataset_summary, COLUMN_NAMES

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
# Download (if needed) and load
train_df, test_df = load_dataset(data_dir='../data/raw', auto_download=True)

print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')
train_df.head(3)

## Dataset Overview

In [ ]:
summary = dataset_summary(train_df)
print('Shape:         ', summary['shape'])
print('Missing values:', summary['missing_values'])
print('Duplicates:    ', summary['duplicate_rows'])
print('Label distribution (train):')
for k, v in summary['label_distribution'].items():
    print(f'  {k}: {v:.1%}')

In [ ]:
# Attack category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary
binary_counts = train_df['binary_label'].value_counts()
axes[0].pie(binary_counts, labels=['Attack', 'Normal'], autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[0].set_title('Binary Label Distribution (Train)')

# Multi-class
cat_counts = train_df['attack_category'].value_counts()
cat_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('husl', len(cat_counts)))
axes[1].set_title('Attack Category Distribution (Train)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/label_distribution.png', dpi=150)
plt.show()

In [ ]:
# Feature type summary
type_summary = train_df.dtypes.value_counts()
print('Data types:')
print(type_summary)

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(f'\nNumeric features:     {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)}')
print(f'Categorical values:   {categorical_cols}')

In [ ]:
# Descriptive statistics for key numeric features
key_features = ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count',
                'serror_rate', 'rerror_rate', 'same_srv_rate', 'dst_host_count']
train_df[key_features].describe().round(2)

In [ ]:
# Protocol type distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(['protocol_type', 'flag', 'service']):
    vc = train_df[col].value_counts().head(15)
    vc.plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2', len(vc)))
    axes[i].set_title(f'{col} distribution')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/categorical_distributions.png', dpi=150)
plt.show()

## Data Dictionary

| # | Feature | Type | Description | Range / Values |
|---|---------|------|-------------|----------------|
| 1 | duration | numeric | Length of connection (seconds) | 0–58329 |
| 2 | protocol_type | categorical | Network protocol | tcp, udp, icmp |
| 3 | service | categorical | Destination service | http, ftp, smtp, … (70 values) |
| 4 | flag | categorical | Connection status | SF, S0, REJ, RSTO, … (11 values) |
| 5 | src_bytes | numeric | Data bytes from source to destination | 0–1.38B |
| 6 | dst_bytes | numeric | Data bytes from destination to source | 0–1.31B |
| 7 | land | binary | 1 if source/destination same host/port | 0, 1 |
| 8 | wrong_fragment | numeric | # of 'wrong' fragments | 0–3 |
| 9 | urgent | numeric | # of urgent packets | 0–14 |
| 10 | hot | numeric | # of hot indicators | 0–101 |
| 11 | num_failed_logins | numeric | # of failed login attempts | 0–5 |
| 12 | logged_in | binary | 1 if successfully logged in | 0, 1 |
| 13 | num_compromised | numeric | # of compromised conditions | 0–7479 |
| 14 | root_shell | binary | 1 if root shell obtained | 0, 1 |
| 15 | su_attempted | binary | 1 if su root command attempted | 0, 1 |
| 16–20 | num_root, num_file_creations, num_shells, num_access_files, num_outbound_cmds | numeric | Operation counts | ≥0 |
| 21 | is_host_login | binary | 1 if login belongs to host list | 0, 1 |
| 22 | is_guest_login | binary | 1 if login is 'guest' | 0, 1 |
| 23 | count | numeric | Connections to same host in last 2s | 0–511 |
| 24 | srv_count | numeric | Connections to same service in last 2s | 0–511 |
| 25–30 | serror_rate, srv_serror_rate, rerror_rate, srv_rerror_rate, same_srv_rate, diff_srv_rate | numeric | Rate-based statistical features | 0.0–1.0 |
| 31 | srv_diff_host_rate | numeric | % connections to different hosts | 0.0–1.0 |
| 32–41 | dst_host_* | numeric | Destination host-level aggregated stats | 0–511 or 0.0–1.0 |
| 42 | label | categorical (target) | Attack type or 'normal' | normal, neptune, smurf, … |
| 43 | binary_label | binary (derived) | 0=normal, 1=attack | 0, 1 |

In [ ]:
# Save full data dictionary to CSV
data_dict = pd.DataFrame({
    'feature': COLUMN_NAMES[:-1],  # exclude difficulty_level
    'dtype': [str(train_df[c].dtype) if c in train_df.columns else 'N/A'
              for c in COLUMN_NAMES[:-1]],
    'n_unique': [train_df[c].nunique() if c in train_df.columns else -1
                 for c in COLUMN_NAMES[:-1]],
    'missing_pct': [train_df[c].isnull().mean() * 100 if c in train_df.columns else -1
                    for c in COLUMN_NAMES[:-1]],
    'example_values': [str(train_df[c].unique()[:3].tolist()) if c in train_df.columns else ''
                       for c in COLUMN_NAMES[:-1]],
})

data_dict.to_csv('../reports/data_dictionary.csv', index=False)
print('Data dictionary saved.')
data_dict.head(10)